# VQA Training Pipeline — Option-Wise Scoring (Corrected)

**Fixes applied:**
1. Removed Phi-2 from training loop entirely (kept for inference only)
2. Replaced broken prefix injection with option-wise scoring
3. Each option is scored independently: `(video + question+option) → score`
4. VQAGuiderCore architecture is **UNCHANGED**

## 1. Install Dependencies

In [ ]:
!pip install torch torchvision opencv-python numpy pillow ftfy regex tqdm
!pip install git+https://github.com/openai/CLIP.git
!pip install transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.7 MB/s eta 0:00:00
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-ke10u5wb
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-ke10u5wb
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=f17fb93ddd805d70b5802d7817774348d578053f0da4463d51f9dde5bc9bbeb5
  Stored in directory: /tmp/pip-ephem-wheel-cache-_or4r6n2/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


## 2. Imports & Device

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import cv2
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Subset
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


## 3. VQAGuiderCore (Unchanged)

In [ ]:
class VQAGuiderCore(nn.Module):
    """Unchanged from testing.ipynb — task planner + tool heads + fusion."""

    def __init__(self, video_dim=512, question_dim=768, hidden_dim=512, num_tasks=3):
        super().__init__()

        # 1. Task Planner
        self.task_planner = nn.Sequential(
            nn.Linear(question_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, num_tasks)
        )

        # 2. Question-Aware Tool Heads
        joint_dim = video_dim + question_dim

        self.action_head = nn.Sequential(
            nn.Linear(joint_dim, 256), nn.ReLU(), nn.LayerNorm(256)
        )
        self.tracking_head = nn.Sequential(
            nn.Linear(joint_dim, 256), nn.ReLU(), nn.LayerNorm(256)
        )
        self.scene_head = nn.Sequential(
            nn.Linear(joint_dim, 256), nn.ReLU(), nn.LayerNorm(256)
        )

        # 3. Fusion
        self.pre_fusion = nn.Sequential(
            nn.Linear(256 * num_tasks + question_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim)
        )

        # 4. Planning Refinement
        self.planning_refine = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim)
        )

        # 5. Output
        self.output_layer = nn.Linear(hidden_dim, 512)

    def forward(self, video_feat, question_feat):
        """
        video_feat:    (B, 512)
        question_feat: (B, 768)
        Returns:
            fusion_vec:  (B, 512)
            task_probs:  (B, num_tasks)
        """
        task_logits = self.task_planner(question_feat)
        task_probs = torch.sigmoid(task_logits)

        joint = torch.cat([video_feat, question_feat], dim=-1)

        action_out = self.action_head(joint)
        tracking_out = self.tracking_head(joint)
        scene_out = self.scene_head(joint)

        tool_outputs = torch.stack(
            [action_out, tracking_out, scene_out], dim=1
        )  # (B, num_tasks, 256)

        weighted_tools = tool_outputs * task_probs.unsqueeze(-1)
        weighted_tools = weighted_tools.view(video_feat.size(0), -1)

        combined = torch.cat([weighted_tools, question_feat], dim=-1)
        fused = self.pre_fusion(combined)
        refined = self.planning_refine(fused)
        fusion_vec = self.output_layer(refined)

        return fusion_vec, task_probs

## 4. LLMProjector (Unchanged)

In [ ]:
class LLMProjector(nn.Module):
    def __init__(self, input_dim=512, llm_dim=2560, num_tokens=10):
        super().__init__()
        self.num_tokens = num_tokens
        self.proj = nn.Linear(input_dim, num_tokens * llm_dim)  # ← Projects to 10x
        self.norm = nn.LayerNorm(llm_dim)                       # ← Fix 3 included!

    def forward(self, x):
        B = x.size(0)
        out = self.proj(x)                        # (B, 10*2560)
        out = out.view(B, self.num_tokens, -1)    # (B, 10, 2560)
        return self.norm(out)                     # (B, 10, 2560) normalized!


## 5. NEW: Option Scorer Head

In [ ]:
class OptionScorer(nn.Module):
    """
    Takes a fusion vector (B, 512) and produces a scalar score.
    Used once per answer option to get option-wise scores.
    """
    def __init__(self, dim=512):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(dim, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        return self.fc(x).squeeze(-1)  # (B,)

## 6. Video Encoder (CLIP + Temporal Attention Pooling)

In [ ]:
def sample_frames(video_path, num_frames=16):
    """Sample evenly spaced frames from a video."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        raise RuntimeError(f"No readable frames in video: {video_path}")

    indices = np.linspace(0, total_frames - 1, num_frames).astype(int)
    frames = []
    frame_id = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_id in indices:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
        frame_id += 1

    cap.release()
    if len(frames) == 0:
        raise RuntimeError(f"Frame extraction failed: {video_path}")
    return frames

In [ ]:
import clip

class CLIPImageEncoder:
    def __init__(self, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model, self.preprocess = clip.load("ViT-B/32", device=self.device)
        self.model.eval()

    def encode(self, frames):
        images = [self.preprocess(Image.fromarray(f)) for f in frames]
        images = torch.stack(images).to(self.device)
        with torch.no_grad():
            embeddings = self.model.encode_image(images)
            embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True)
        return embeddings  # (N, 512)


class TemporalAttentionPooling(nn.Module):
    def __init__(self, dim=512):
        super().__init__()
        self.proj = nn.Linear(dim, dim)
        self.attn_fc = nn.Sequential(
            nn.Linear(dim, dim // 4),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(dim // 4, 1)
        )

    def forward(self, x):  # x: (N, 512)
        x = x.float()
        proj_x = self.proj(x)
        attn_logits = self.attn_fc(proj_x)
        weights = torch.softmax(attn_logits, dim=0).squeeze(-1)
        pooled = (weights.unsqueeze(-1) * x).sum(dim=0)
        return pooled, weights


class VideoEncoder:
    def __init__(self, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.frame_encoder = CLIPImageEncoder(self.device)
        self.pooling = TemporalAttentionPooling().to(self.device)

    def encode(self, video_path, num_frames=16):
        frames = sample_frames(video_path, num_frames)
        frame_embeddings = self.frame_encoder.encode(frames)
        video_embedding, weights = self.pooling(frame_embeddings)
        return video_embedding

## 7. Question Encoder (DistilBERT)

In [ ]:
from transformers import DistilBertTokenizer, DistilBertModel

_q_tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
_q_model = DistilBertModel.from_pretrained("distilbert-base-uncased")

_q_model.to(device)   # ← THIS WAS MISSING
_q_model.eval()

for p in _q_model.parameters():
    p.requires_grad = False


def get_question_option_embedding(question: str, option: str):
    text = f"{question} [SEP] {option}"
    inputs = _q_tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = _q_model(**inputs)

    return outputs.last_hidden_state[:, 0, :].squeeze(0)

def get_question_embedding(question: str):
    inputs = _q_tokenizer(
        question,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = _q_model(**inputs)

    return outputs.last_hidden_state[:, 0, :].squeeze(0)


print("DistilBERT loaded ✅")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DistilBERT loaded ✅


## 8. Dataset & Video Index

In [ ]:
def build_video_index(video_root):
    """Scan video_root for videos, returning {video_id: full_path}."""
    video_map = {}
    for folder in os.listdir(video_root):
        folder_path = os.path.join(video_root, folder)
        if not os.path.isdir(folder_path):
            continue
        for file in os.listdir(folder_path):
            if file.endswith((".mp4", ".avi", ".webm")):
                video_id = os.path.splitext(file)[0]
                video_map[video_id] = os.path.join(folder_path, file)
    return video_map


class NextQADataset(Dataset):
    """
    NExT-QA dataset filtered to only include videos that exist in video_map.
    Returns: (video_path, question, [a0..a4], answer_index)
    """
    def __init__(self, csv_file, video_map):
        full_data = pd.read_csv(csv_file)
        self.video_map = video_map

        # Filter to only rows whose video exists
        # ✅ FIXED — filters by your cache (700+ videos!)
        available_ids = set(video_features.keys())   # ← Use cache keys, not video_map!
        self.data = full_data[
            full_data["video"].astype(int).astype(str).isin(available_ids)
        ].reset_index(drop=True)
        self.data = self.data.head(200)


        print(f"Dataset: {len(self.data)} rows (from {len(full_data)} total, "
              f"{len(available_ids)} videos available)")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        video_id = str(int(row["video"]))
        video_path = self.video_map[video_id]
        question = row["question"]
        options = [row["a0"], row["a1"], row["a2"], row["a3"], row["a4"]]
        answer = int(row["answer"])
        return video_path, question, options, answer

## 9. Mount Drive & Setup Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Adjust these paths for your setup ──
CSV_PATH = "/content/drive/MyDrive/dataset/train.csv"
VIDEO_ROOT = "/content/drive/MyDrive/dataset/Training"
CACHE_PATH = "/content/drive/MyDrive/dataset/video_features.pt"
SAVE_PATH = "/content/drive/MyDrive/dataset/first/vqa_model_best.pt"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 10. Build Video Index & Dataset

In [ ]:
video_features = torch.load(CACHE_PATH, map_location="cpu")
print(f"Cache loaded: {len(video_features)} videos")
# Build a dummy video_map from cache keys (for dataset compatibility)
video_map = {vid_id: vid_id for vid_id in video_features.keys()}


# Now dataset will see ALL 700+ cached videos!
dataset = NextQADataset(CSV_PATH, video_map)
print(f"Dataset now uses {len(dataset)} rows from {len(video_features)} cached videos!")


val_size = int(0.2 * len(dataset))
train_size = len(dataset) - val_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])
print(f"Train: {train_size} samples | Val: {val_size} samples")

print(f"\nSample row:")
print(dataset[0])

Cache loaded: 721 videos
Dataset: 200 rows (from 34132 total, 721 videos available)
Dataset now uses 200 rows from 721 cached videos!
Train: 160 samples | Val: 40 samples

Sample row:
('3238737531', 'how many children are in the video', ['one', 'three', 'seven', 'two', 'five'], 3)


## 11. Pre-cache Video Features

In [ ]:
def precompute_video_features(dataset, video_encoder, cache_path=None):
    """
    Encode all unique videos once and cache them.
    Returns: {video_path: tensor (512,)}
    """
    # When loading the locally-extracted cache, no need to re-extract:
    if cache_path and os.path.exists(cache_path):
        print(f"Loading cached video features from {cache_path}")
        features = torch.load(cache_path, map_location="cpu")
        # Check if keys are IDs (local extraction) or paths (Colab extraction)
        sample_key = next(iter(features))
        print(f"Cache key format: '{sample_key}'")   # Will print "13884124143"
        return features


    print("Pre-computing video features...")
    video_features = {}
    unique_paths = set()

    for i in range(len(dataset)):
        unique_paths.add(dataset[i][0])

    for i, vp in enumerate(unique_paths):
        feat = video_encoder.encode(vp)
        if not isinstance(feat, torch.Tensor):
            feat = torch.tensor(feat)
        if feat.dim() == 2:
            feat = feat.squeeze(0)
        video_features[vp] = feat.detach().cpu()
        print(f"  [{i+1}/{len(unique_paths)}] {os.path.basename(vp)}")

    if cache_path:
        torch.save(video_features, cache_path)
        print(f"Saved cache → {cache_path}")

    return video_features


video_encoder = VideoEncoder()
video_features = precompute_video_features(dataset, video_encoder, cache_path=CACHE_PATH)

100%|████████████████████████████████████████| 338M/338M [00:02<00:00, 118MiB/s]


Loading cached video features from /content/drive/MyDrive/dataset/video_features.pt
Cache key format: '3238737531'


## 12. Collate Function (Option-Wise Encoding)

In [ ]:
precomputed_qo = {}

for video_path, question, options, _ in dataset:
    video_id = os.path.splitext(os.path.basename(video_path))[0]

    opt_feats = []
    for opt in options:
        qo_emb = get_question_option_embedding(question, str(opt))
        opt_feats.append(qo_emb.detach().cpu())

    precomputed_qo[video_id] = torch.stack(opt_feats)  # (5, 768)

def collate_fn(batch):
    video_feats = []
    option_feats_list = []
    answers = []
    input_ids_list = []
    attn_mask_list = []

    for video_path, question, options, answer in batch:
        video_id = os.path.splitext(os.path.basename(video_path))[0]

        # CPU only
        vf = video_features[video_id].clone().detach()
        video_feats.append(vf)

        video_id = os.path.splitext(os.path.basename(video_path))[0]
        opt_feats = precomputed_qo[video_id]
        option_feats_list.append(opt_feats)

        answers.append(answer)

        answer_text = str(options[answer])
        full_text = (
            f"You are a video understanding AI.\n"
            f"Question: {question}\n"
            f"Detailed Answer: The answer is {answer_text}."
        )

        enc = phi2_tokenizer(
            full_text,
            return_tensors="pt",
            truncation=True,
            max_length=64,
            padding="max_length"
        )

        input_ids_list.append(enc.input_ids.squeeze(0))
        attn_mask_list.append(enc.attention_mask.squeeze(0))

    return {
        "video_feat": torch.stack(video_feats),
        "option_feats": torch.stack(option_feats_list),
        "answer": torch.tensor(answers, dtype=torch.long),
        "input_ids": torch.stack(input_ids_list),
        "attn_mask": torch.stack(attn_mask_list),
    }

## 13. DataLoader

In [ ]:
BATCH_SIZE = 4

train_loader = DataLoader(
    train_dataset,          # ← Use train_dataset, NOT dataset
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_dataset,            # ← Separate validation loader
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

Train batches: 40 | Val batches: 10


## 14. Training Step (Option-Wise Scoring)

In [ ]:
def train_step(batch, vqaguider, projector, scorer, optimizer):
    """
    Option-wise scoring:
      For each of the 5 options:
        1. Run VQAGuiderCore(video_feat, question+option_feat) → fusion_vec
        2. Run LLMProjector(fusion_vec)
        3. Run OptionScorer(fusion_vec) → scalar score
      Stack scores → (B, 5) → CrossEntropy
    """

    video_feat   = batch["video_feat"].to(device)
    option_feats = batch["option_feats"].to(device)
    answers      = batch["answer"].to(device)

    B = video_feat.size(0)
    num_options = 5

    all_scores = []
    all_task_probs = []

    for opt_idx in range(num_options):
        # Question+option embedding for this option
        qo_feat = option_feats[:, opt_idx, :]  # (B, 768)

        # VQAGuiderCore forward
        fusion_vec, task_probs = vqaguider(video_feat, qo_feat)  # (B, 512), (B, 3)

        # Score this option
        score = scorer(fusion_vec)  # (B,)

        all_scores.append(score)
        all_task_probs.append(task_probs)

    # Stack scores → (B, 5)
    option_scores = torch.stack(all_scores, dim=1)  # (B, 5)

    # Average task probs across options for entropy regularization
    avg_task_probs = torch.stack(all_task_probs, dim=0).mean(dim=0)  # (B, 3)

    # ─── Losses ──────────────────────────────────────────────────────────
    loss_mcq = F.cross_entropy(option_scores, answers)
    entropy = -(avg_task_probs * torch.log(avg_task_probs + 1e-8)).sum(dim=-1).mean()
    sparsity_penalty = avg_task_probs.sum(dim=-1).mean()  # ← ADD THIS LINE
    # Total loss — must match validate() exactly!
    loss = loss_mcq + 0.01 * entropy + 0.05 * sparsity_penalty

    # ─── Backprop ────────────────────────────────────────────────────────
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # ─── Accuracy ────────────────────────────────────────────────────────
    preds = option_scores.argmax(dim=1)
    acc = (preds == answers).float().mean().item()

    return loss.item(), acc

## 15. Validation

In [ ]:
@torch.no_grad()
def validate(dataloader, vqaguider, projector, scorer):
    """Run validation and return average loss + accuracy."""
    vqaguider.eval()
    projector.eval()
    scorer.eval()

    total_loss = 0
    total_acc = 0
    n_batches = 0

    for batch in dataloader:
        video_feat   = batch["video_feat"].to(device)
        option_feats = batch["option_feats"].to(device)
        answers      = batch["answer"].to(device)

        all_scores = []
        all_task_probs = []

        for opt_idx in range(5):
            qo_feat = option_feats[:, opt_idx, :]
            fusion_vec, task_probs = vqaguider(video_feat, qo_feat)
            score = scorer(fusion_vec)
            all_scores.append(score)
            all_task_probs.append(task_probs)

        option_scores = torch.stack(all_scores, dim=1)
        avg_task_probs = torch.stack(all_task_probs, dim=0).mean(dim=0)

        loss_mcq = F.cross_entropy(option_scores, answers)
        entropy = -(avg_task_probs * torch.log(avg_task_probs + 1e-8)).sum(dim=-1).mean()
        sparsity_penalty = avg_task_probs.sum(dim=-1).mean()
        loss = loss_mcq + 0.01 * entropy + 0.05 * sparsity_penalty

        preds = option_scores.argmax(dim=1)
        acc = (preds == answers).float().mean().item()

        total_loss += loss.item()
        total_acc += acc
        n_batches += 1

    vqaguider.train()
    projector.train()
    scorer.train()

    return total_loss / max(n_batches, 1), total_acc / max(n_batches, 1)

## 16. Initialize Models & Optimizer

In [ ]:
vqaguider = VQAGuiderCore().to(device)
projector = LLMProjector(512, 2560).to(device)
scorer = OptionScorer(512).to(device)

print(f"Device: {device}")
print(f"Trainable params:")
print(f"  VQAGuiderCore: {sum(p.numel() for p in vqaguider.parameters()):,}")
print(f"  LLMProjector:  {sum(p.numel() for p in projector.parameters()):,}")
print(f"  OptionScorer:  {sum(p.numel() for p in scorer.parameters()):,}")
total = (sum(p.numel() for p in vqaguider.parameters()) +
         sum(p.numel() for p in projector.parameters()) +
         sum(p.numel() for p in scorer.parameters()))
print(f"  Total:         {total:,}")

LR = 1e-4

optimizer = torch.optim.Adam(
    list(vqaguider.parameters()) +
    list(scorer.parameters()),      # ← scorer YES, projector NO
    lr=LR,
    weight_decay=1e-5,
)

Device: cuda
Trainable params:
  VQAGuiderCore: 2,694,915
  LLMProjector:  13,137,920
  OptionScorer:  131,585
  Total:         15,964,420


## 17. 🚀 Training Loop

In [ ]:
EPOCHS = 30
VAL_EVERY = 5

print("=" * 70)
print("  VQA Training Pipeline — Option-Wise Scoring")
print("=" * 70)
print(f"Epochs: {EPOCHS}, Batch size: {BATCH_SIZE}, LR: {LR}")
print("-" * 70)

best_acc = 0.0

for epoch in range(EPOCHS):
    vqaguider.train()
    projector.train()
    scorer.train()

    epoch_loss = 0.0
    epoch_acc = 0.0
    n_batches = 0

    for batch in train_loader:
        loss, acc = train_step(batch, vqaguider, projector, scorer, optimizer)
        epoch_loss += loss
        epoch_acc += acc
        n_batches += 1

    avg_loss = epoch_loss / max(n_batches, 1)
    avg_acc = epoch_acc / max(n_batches, 1)

    print(f"Epoch {epoch+1:3d}/{EPOCHS} │ Loss: {avg_loss:.4f} │ Acc: {avg_acc:.4f}")

    # ─── Validation ──────────────────────────────────────────────────
    if (epoch + 1) % VAL_EVERY == 0:
        # ← Change train_loader to val_loader here!
        val_loss, val_acc = validate(val_loader, vqaguider, projector, scorer)
        print(f"          ↳ Val Loss: {val_loss:.4f} │ Val Acc: {val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save({
                "vqaguider": vqaguider.state_dict(),
                "projector": projector.state_dict(),
                "scorer": scorer.state_dict(),
                "epoch": epoch + 1,
                "best_acc": best_acc,
            }, SAVE_PATH)
            print(f"          ↳ Saved best model → {SAVE_PATH}")

print("-" * 70)
print(f"Training complete! Best accuracy: {best_acc:.4f}")

  VQA Training Pipeline — Option-Wise Scoring
Epochs: 30, Batch size: 4, LR: 0.0001
----------------------------------------------------------------------
Epoch   1/30 │ Loss: 1.0055 │ Acc: 0.6250
Epoch   2/30 │ Loss: 1.1540 │ Acc: 0.5687
Epoch   3/30 │ Loss: 0.8945 │ Acc: 0.6562
Epoch   4/30 │ Loss: 0.7990 │ Acc: 0.7000
Epoch   5/30 │ Loss: 0.7241 │ Acc: 0.7125
          ↳ Val Loss: 1.9880 │ Val Acc: 0.2250
          ↳ Saved best model → /content/drive/MyDrive/dataset/first/vqa_model_best.pt
Epoch   6/30 │ Loss: 0.7259 │ Acc: 0.7562
Epoch   7/30 │ Loss: 0.5771 │ Acc: 0.7625
Epoch   8/30 │ Loss: 0.5635 │ Acc: 0.7812
Epoch   9/30 │ Loss: 0.4519 │ Acc: 0.8187
Epoch  10/30 │ Loss: 0.4004 │ Acc: 0.8500
          ↳ Val Loss: 4.2077 │ Val Acc: 0.2250
Epoch  11/30 │ Loss: 0.3283 │ Acc: 0.8562
Epoch  12/30 │ Loss: 0.3874 │ Acc: 0.8375
Epoch  13/30 │ Loss: 0.3962 │ Acc: 0.8438
Epoch  14/30 │ Loss: 0.3264 │ Acc: 0.8938
Epoch  15/30 │ Loss: 0.2430 │ Acc: 0.9000
          ↳ Val Loss: 4.6270 │ Val 

## 18. Load Checkpoint (for later use)

In [ ]:
# To load the best checkpoint later:
# ckpt = torch.load(SAVE_PATH)
# vqaguider.load_state_dict(ckpt["vqaguider"])
# projector.load_state_dict(ckpt["projector"])
# scorer.load_state_dict(ckpt["scorer"])
# print(f"Loaded checkpoint from epoch {ckpt['epoch']}, acc={ckpt['best_acc']:.4f}")

In [ ]:
!pip install -q transformers==4.37.2 accelerate

In [ ]:
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
phi2_name = "microsoft/phi-2"
phi2_tokenizer = AutoTokenizer.from_pretrained(phi2_name, trust_remote_code=True)
phi2_tokenizer.pad_token = phi2_tokenizer.eos_token
phi2 = AutoModelForCausalLM.from_pretrained(
    phi2_name,
    torch_dtype=torch.float16,
    trust_remote_code=True
).to(device)
phi2.eval()
for p in phi2.parameters():
    p.requires_grad = False
print("Phi-2 loaded ✅")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Phi-2 loaded ✅


In [ ]:

import time
# ── Step 0: Load Phase 1 trained weights ──────────────────────
PHASE1_PATH = "/content/drive/MyDrive/dataset/1/vqa_model_best.pt"
print("Loading Phase 1 checkpoint...")
ckpt = torch.load(PHASE1_PATH, map_location=device)
vqaguider.load_state_dict(ckpt["vqaguider"])
# ⚠️ DO NOT load projector from Phase 1 — architecture changed!
scorer.load_state_dict(ckpt["scorer"])
print(f"  ✅ Loaded (epoch {ckpt.get('epoch','?')}, acc={ckpt.get('best_acc','?')})")
print("  ℹ️  Projector NOT loaded — new 10-token architecture, trains from scratch")

# ── Step 1: Freeze everything except projector ──────────────────
vqaguider.eval()
for p in vqaguider.parameters():
    p.requires_grad = False

scorer.eval()
for p in scorer.parameters():
    p.requires_grad = False

projector.train()
proj_optimizer = torch.optim.Adam(projector.parameters(), lr=5e-5)

PHASE2_EPOCHS = 10

# ── Create small fast Phase 2 loader ────────────────────────────
from torch.utils.data import Subset
phase2_subset = Subset(dataset, range(min(100, len(dataset))))
phase2_loader = DataLoader(
    phase2_subset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)
print(f"Phase 2 subset: {len(phase2_subset)} samples | Epochs: {PHASE2_EPOCHS}")

SAVE_PATH_GEN = "/content/drive/MyDrive/dataset/1/vqa_model_generative.pt"

print("=" * 60)
print("  Phase 2: Teaching projector to work with Phi-2")
print("=" * 60)

for epoch in range(PHASE2_EPOCHS):
    projector.train()
    epoch_loss = 0.0
    count = 0

    for batch in tqdm(phase2_loader, desc=f"Epoch {epoch+1}/{PHASE2_EPOCHS}"):
        start = time.time()

        # your training code here

        video_feat   = batch["video_feat"].to(device)
        option_feats = batch["option_feats"].to(device)
        answers      = batch["answer"].to(device)
        input_ids    = batch["input_ids"].to(device)
        attn_mask    = batch["attn_mask"].to(device)

        B = video_feat.size(0)

        # ✅ Get correct option embedding
        correct_qo_feat = option_feats[torch.arange(B, device=device), answers, :]

        # ✅ Get fusion vector (frozen)
        with torch.no_grad():
            fusion_vec, _ = vqaguider(video_feat, correct_qo_feat)

        # ✅ Project to prefix
        prefix = projector(fusion_vec).to(dtype=torch.float16)   # (B, 10, 2560)

        # ✅ Get token embeddings (BATCHED!)
        with torch.no_grad():
            token_embeds = phi2.get_input_embeddings()(input_ids)  # (B, T, 2560)
            token_embeds = token_embeds.to(dtype=torch.float16)

        # ✅ Concatenate prefix + tokens
        inputs_embeds = torch.cat([prefix, token_embeds], dim=1)   # (B, 10+T, 2560)

        # ✅ Attention mask
        prefix_mask = torch.ones(
            (B, projector.num_tokens),
            dtype=attn_mask.dtype,
            device=device
        )
        attention_mask = torch.cat([prefix_mask, attn_mask], dim=1)

        # ✅ Labels (mask prefix)
        prefix_labels = torch.full(
            (B, projector.num_tokens),
            -100,
            dtype=input_ids.dtype,
            device=device
        )
        labels = torch.cat([prefix_labels, input_ids], dim=1)

        # ✅ Forward pass (SINGLE CALL!)
        outputs = phi2(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        proj_optimizer.zero_grad()
        loss.backward()
        proj_optimizer.step()

        epoch_loss += loss.item()
        count += 1
        print("Batch time:", time.time() - start)


    avg_loss = epoch_loss / max(count, 1)
    print(f"Epoch {epoch+1:3d}/{PHASE2_EPOCHS} | Loss: {avg_loss:.4f}")

# Save everything
torch.save({
    "vqaguider": vqaguider.state_dict(),
    "projector": projector.state_dict(),
    "scorer":    scorer.state_dict(),
}, SAVE_PATH_GEN)
print(f"\nSaved → {SAVE_PATH_GEN}")
print("Phase 2 done! ✅")


In [ ]:
projector.eval()
test_video = "/content/drive/MyDrive/dataset/Training/0000/3238737531.mp4"
test_question = "how many children are in that video and what are children doing in that video?"

vf = video_encoder.encode(test_video).unsqueeze(0).to(device)
qf = get_question_embedding(test_question).unsqueeze(0).to(device)

with torch.no_grad():
    fusion_vec, task_probs = vqaguider(vf, qf)              # ✅ capture task_probs
    prefix = projector(fusion_vec).to(dtype=torch.float16)  # (1, 10, 2560)

    prompt = (
        f"You are a video understanding AI. Watch the video carefully and answer.\n"
        f"Question: {test_question}\nDetailed Answer:"
    )
    enc = phi2_tokenizer(prompt, return_tensors="pt").to(device)
    token_embeds = phi2.get_input_embeddings()(enc.input_ids)
    inputs_embeds = torch.cat([prefix, token_embeds], dim=1)

    prefix_mask = torch.ones(                               # ✅ 10 tokens
        (1, projector.num_tokens),
        dtype=enc.attention_mask.dtype, device=device
    )
    attention_mask = torch.cat([prefix_mask, enc.attention_mask], dim=1)

    output = phi2.generate(
        inputs_embeds=inputs_embeds,
        attention_mask=attention_mask,
        max_new_tokens=50,
        do_sample=False,
        pad_token_id=phi2_tokenizer.eos_token_id,
    )

answer = phi2_tokenizer.decode(output[0], skip_special_tokens=True)
answer = answer.split("Detailed Answer:")[-1].strip()       # ✅ correct split
answer = answer.split("\n")[0].strip()

tp = task_probs.squeeze().tolist()
print("=" * 60)
print(f"  Question : {test_question}")                      # ✅ test_question
print(f"  Answer   : {answer}")
print(f"  Task Routing:")
print(f"    Action   : {'█' * int(tp[0]*20)} {tp[0]:.2f}")
print(f"    Tracking : {'█' * int(tp[1]*20)} {tp[1]:.2f}")
print(f"    Scene    : {'█' * int(tp[2]*20)} {tp[2]:.2f}")
print("=" * 60)


  Question : how many children are in that video and what are children doing in that video?
  Answer   : There are two children in that video. One child is playing with a ball and the other child is dancing.
  Task Routing:
    Action   :  0.00
    Tracking :  0.00
    Scene    :  0.00


In [ ]:
# 🔥 IMPORTANT: Load GENERATIVE model (Phase 2)
ckpt = torch.load("/content/drive/MyDrive/dataset/vqa_model_generative.pt", map_location=device)

vqaguider.load_state_dict(ckpt["vqaguider"])
scorer.load_state_dict(ckpt["scorer"])
projector.load_state_dict(ckpt["projector"])

vqaguider.to(device).eval()
scorer.to(device).eval()
projector.to(device).eval()
phi2.to(device).eval()


# ─────────────────────────────────────────────

test_video = "/content/drive/MyDrive/dataset/Training/0000/3238737531.mp4"
test_question = "how many children are in the video"

# ✅ Encode video
vf = video_encoder.encode(test_video).unsqueeze(0).to(device)

# ✅ IMPORTANT: Use question+option embedding (NOT question alone)
dummy_option = "some activity happening in the video"
qf = get_question_option_embedding(test_question, dummy_option).unsqueeze(0).to(device)

with torch.no_grad():
    # ✅ Get fusion + task routing
    fusion_vec, task_probs = vqaguider(vf, qf)

    # ✅ Project to prefix
    prefix = projector(fusion_vec).to(dtype=torch.float16)  # (1, 10, 2560)

    # ✅ Prompt
    prompt = (
        "You are a video understanding AI. Watch the video carefully and answer.\n"
        f"Question: {test_question}\n"
        "Detailed Answer:"
    )

    enc = phi2_tokenizer(prompt, return_tensors="pt").to(device)

    # ✅ Token embeddings
    token_embeds = phi2.get_input_embeddings()(enc.input_ids)
    token_embeds = token_embeds.to(dtype=torch.float16)

    # ✅ Combine prefix + tokens
    inputs_embeds = torch.cat([prefix, token_embeds], dim=1)

    # ✅ Attention mask
    prefix_mask = torch.ones(
        (1, projector.num_tokens),
        dtype=enc.attention_mask.dtype,
        device=device
    )
    attention_mask = torch.cat([prefix_mask, enc.attention_mask], dim=1)

    # ✅ Generate
    output = phi2.generate(
        inputs_embeds=inputs_embeds,
        attention_mask=attention_mask,
        max_new_tokens=50,
        do_sample=False,
        pad_token_id=phi2_tokenizer.eos_token_id,
    )

# ✅ Decode
answer = phi2_tokenizer.decode(output[0], skip_special_tokens=True)
answer = answer.split("Detailed Answer:")[-1].strip()
answer = answer.split("\n")[0].strip()

# ✅ Task routing
print("RAW task_probs:", task_probs)
tp = task_probs.squeeze().tolist()

print("=" * 60)
print(f"  Question : {test_question}")
print(f"  Answer   : {answer}")
print(f"  Task Routing:")
print(f"    Action   : {'█' * int(tp[0]*20)} {tp[0]:.2f}")
print(f"    Tracking : {'█' * int(tp[1]*20)} {tp[1]:.2f}")
print(f"    Scene    : {'█' * int(tp[2]*20)} {tp[2]:.2f}")
print("=" * 60)

RAW task_probs: tensor([[0.0010, 0.0023, 0.0005]], device='cuda:0')
  Question : how many children are in the video
  Answer   : The answer is two.
  Task Routing:
    Action   :  0.00
    Tracking :  0.00
    Scene    :  0.00


**BASE PIPELINE**

In [ ]:
"""
=============================================================
BASELINE VQA PIPELINE — Simple Concatenation (No Routing)
=============================================================
Purpose: Benchmark comparison against VQAGuider.
This pipeline is IDENTICAL to VQAGuider EXCEPT:
  - VQAGuiderCore (Task Planner + Tool Heads) is REMOVED
  - Replaced with a simple MLP that just concatenates video + question
  - No task probabilities, no routing, no entropy loss
  - Same dataset, same video features, same OptionScorer
Expected outcome: VQAGuider should outperform this baseline,
proving that Explicit Neural Routing adds real value.
=============================================================
"""

In [ ]:
# ── Cell 1: Install Dependencies ───────────────────────────
!pip install torch torchvision opencv-python numpy pillow ftfy regex tqdm
!pip install git+https://github.com/openai/CLIP.git
!pip install transformers

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-ten_jhhk
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-ten_jhhk
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done


In [ ]:
# ── Cell 2: Imports & Device ───────────────────────────────
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import cv2
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
# ── Cell 3: BASELINE MODEL (Replaces VQAGuiderCore) ────────
class BaselineVQA(nn.Module):
    """
    Simple concatenation baseline — NO routing, NO tool heads.
    Architecture:
        concat(video_512, question_768) → 1280
        → Linear(1280, 512) → ReLU → Dropout
        → Linear(512, 512)  → ReLU → LayerNorm
        → Linear(512, 512)  [output]
    This is the "dumb" version of VQAGuider.
    Same input/output dimensions so the OptionScorer is reusable.
    """
    def __init__(self, video_dim=512, question_dim=768, hidden_dim=512):
        super().__init__()
        joint_dim = video_dim + question_dim  # 1280
        # Simple 3-layer MLP — NO task planner, NO expert heads
        self.fusion = nn.Sequential(
            nn.Linear(joint_dim, hidden_dim),   # 1280 → 512
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),  # 512 → 512
            nn.ReLU(),
            nn.LayerNorm(hidden_dim)
        )
        self.output_layer = nn.Linear(hidden_dim, 512)  # 512 → 512
    def forward(self, video_feat, question_feat):
        """
        video_feat:    (B, 512)
        question_feat: (B, 768)
        Returns:
            fusion_vec: (B, 512)   — same shape as VQAGuiderCore output
        """
        combined = torch.cat([video_feat, question_feat], dim=-1)  # (B, 1280)
        fused = self.fusion(combined)                               # (B, 512)
        out = self.output_layer(fused)                              # (B, 512)
        return out   # No task_probs — that's the whole point!

In [ ]:
# ── Cell 4: Option Scorer (IDENTICAL to VQAGuider) ─────────
class OptionScorer(nn.Module):
    """
    Exactly the same as VQAGuider's OptionScorer.
    Takes fusion_vec (B, 512) → scalar score (B,)
    """
    def __init__(self, dim=512):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(dim, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 1)
        )
    def forward(self, x):
        return self.fc(x).squeeze(-1)  # (B,)

In [ ]:
# ── Cell 5: Video Encoder (IDENTICAL to VQAGuider) ─────────
def sample_frames(video_path, num_frames=16):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        raise RuntimeError(f"No frames: {video_path}")
    indices = np.linspace(0, total_frames - 1, num_frames).astype(int)
    frames, frame_id = [], 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_id in indices:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        frame_id += 1
    cap.release()
    return frames
import clip
class CLIPImageEncoder:
    def __init__(self, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model, self.preprocess = clip.load("ViT-B/32", device=self.device)
        self.model.eval()
    def encode(self, frames):
        images = torch.stack([self.preprocess(Image.fromarray(f)) for f in frames]).to(self.device)
        with torch.no_grad():
            emb = self.model.encode_image(images)
            emb = emb / emb.norm(dim=-1, keepdim=True)
        return emb
class TemporalAttentionPooling(nn.Module):
    def __init__(self, dim=512):
        super().__init__()
        self.proj = nn.Linear(dim, dim)
        self.attn_fc = nn.Sequential(
            nn.Linear(dim, dim // 4), nn.ReLU(), nn.Dropout(0.1), nn.Linear(dim // 4, 1)
        )
    def forward(self, x):
        x = x.float()
        proj_x = self.proj(x)
        weights = torch.softmax(self.attn_fc(proj_x), dim=0).squeeze(-1)
        return (weights.unsqueeze(-1) * x).sum(dim=0), weights
class VideoEncoder:
    def __init__(self, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.frame_encoder = CLIPImageEncoder(self.device)
        self.pooling = TemporalAttentionPooling().to(self.device)
    def encode(self, video_path, num_frames=16):
        frames = sample_frames(video_path, num_frames)
        frame_embeddings = self.frame_encoder.encode(frames)
        video_embedding, _ = self.pooling(frame_embeddings)
        return video_embedding

In [ ]:
# ── Cell 6: Question Encoder (IDENTICAL to VQAGuider) ──────
from transformers import DistilBertTokenizer, DistilBertModel
_q_tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
_q_model = DistilBertModel.from_pretrained("distilbert-base-uncased").to(device)
_q_model.eval()
for p in _q_model.parameters():
    p.requires_grad = False
def get_question_option_embedding(question: str, option: str):
    inputs = _q_tokenizer(
        f"{question} [SEP] {option}",
        return_tensors="pt", padding=True, truncation=True, max_length=128
    ).to(device)
    with torch.no_grad():
        outputs = _q_model(**inputs)
    return outputs.last_hidden_state[:, 0, :].squeeze(0)
print("DistilBERT loaded ✅")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DistilBERT loaded ✅


In [ ]:
# ── Cell 7: Dataset (IDENTICAL to VQAGuider) ───────────────
def build_video_index(video_root):
    video_map = {}
    for folder in os.listdir(video_root):
        folder_path = os.path.join(video_root, folder)
        if not os.path.isdir(folder_path):
            continue
        for file in os.listdir(folder_path):
            if file.endswith((".mp4", ".avi", ".webm")):
                video_id = os.path.splitext(file)[0]
                video_map[video_id] = os.path.join(folder_path, file)
    return video_map
class NextQADataset(Dataset):
    def __init__(self, csv_file, video_map, max_samples=200):
        full_data = pd.read_csv(csv_file)
        # ✅ FIXED — filters by your cache (700+ videos!)
        available_ids = set(video_features.keys())   # ← Use cache keys, not video_map!
        self.data = full_data[
            full_data["video"].astype(int).astype(str).isin(available_ids)
        ].reset_index(drop=True)
        self.data = self.data.head(200)
        self.video_map = video_map
        print(f"Dataset: {len(self.data)} rows")
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        video_id = str(int(row["video"]))
        return (
            self.video_map[video_id],
            row["question"],
            [row["a0"], row["a1"], row["a2"], row["a3"], row["a4"]],
            int(row["answer"])
        )

In [ ]:
# ── Cell 8: Mount Drive & Paths ────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
CSV_PATH    = "/content/drive/MyDrive/dataset/train.csv"
VIDEO_ROOT  = "/content/drive/MyDrive/dataset/Training"
CACHE_PATH  = "/content/drive/MyDrive/dataset/video_features.pt"   # ← SAME CACHE as VQAGuider!
BASELINE_SAVE = "/content/drive/MyDrive/dataset/baseline_model_best.pt"

Mounted at /content/drive


In [ ]:
# ── Cell 9: Build Index & Split ────────────────────────────


video_features = torch.load(CACHE_PATH, map_location="cpu")
print(f"Cache loaded: {len(video_features)} videos")
# Build a dummy video_map from cache keys (for dataset compatibility)
video_map = {vid_id: vid_id for vid_id in video_features.keys()}


# Now dataset will see ALL 700+ cached videos!
dataset = NextQADataset(CSV_PATH, video_map)
print(f"Dataset now uses {len(dataset)} rows from {len(video_features)} cached videos!")


val_size = int(0.2 * len(dataset))
train_size = len(dataset) - val_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])
print(f"Train: {train_size} samples | Val: {val_size} samples")

print(f"\nSample row:")
print(dataset[0])

Cache loaded: 721 videos
Dataset: 200 rows
Dataset now uses 200 rows from 721 cached videos!
Train: 160 samples | Val: 40 samples

Sample row:
('3238737531', 'how many children are in the video', ['one', 'three', 'seven', 'two', 'five'], 3)


In [ ]:
# ── Cell 10: Load Video Feature Cache ──────────────────────
# Uses the EXACT SAME cache extracted by VQAGuider!
# This guarantees a fair comparison — same visual input for both models.
print(f"Loading cached video features from {CACHE_PATH}")
video_features = torch.load(CACHE_PATH, map_location="cpu")
print(f"Cache loaded: {len(video_features)} videos ✅")

Loading cached video features from /content/drive/MyDrive/dataset/video_features.pt
Cache loaded: 721 videos ✅


In [ ]:
# ── Cell 11: Pre-compute Question Embeddings ────────────────
print("Pre-computing question+option embeddings...")
precomputed_qo = {}
for video_path, question, options, _ in tqdm(dataset):
    video_id = os.path.splitext(os.path.basename(video_path))[0]
    if video_id not in precomputed_qo:
        opt_feats = []
        for opt in options:
            qo_emb = get_question_option_embedding(question, str(opt))
            opt_feats.append(qo_emb.detach().cpu())
        precomputed_qo[video_id] = torch.stack(opt_feats)  # (5, 768)
print(f"Pre-computed embeddings for {len(precomputed_qo)} videos ✅")

Pre-computing question+option embeddings...


100%|██████████| 200/200 [00:04<00:00, 40.88it/s]

Pre-computed embeddings for 196 videos ✅


In [ ]:
# ── Cell 12: Collate Function ──────────────────────────────
def collate_fn(batch):
    video_feats, option_feats_list, answers = [], [], []
    for video_path, question, options, answer in batch:
        video_id = os.path.splitext(os.path.basename(video_path))[0]
        video_feats.append(video_features[video_id].clone().detach())
        option_feats_list.append(precomputed_qo[video_id])
        answers.append(answer)
    return {
        "video_feat":   torch.stack(video_feats),
        "option_feats": torch.stack(option_feats_list),
        "answer":       torch.tensor(answers, dtype=torch.long),
    }

In [ ]:
# ── Cell 13: DataLoaders ────────────────────────────────────
BATCH_SIZE = 4
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

Train batches: 40 | Val batches: 10


In [ ]:
# ── Cell 14: Baseline Training Step ────────────────────────
def baseline_train_step(batch, baseline_model, scorer, optimizer):
    """
    KEY DIFFERENCE from VQAGuider:
    - No task probabilities
    - No entropy loss
    - No sparsity penalty
    - Just pure cross-entropy on option scores
    """
    video_feat   = batch["video_feat"].to(device)    # (B, 512)
    option_feats = batch["option_feats"].to(device)  # (B, 5, 768)
    answers      = batch["answer"].to(device)         # (B,)
    B = video_feat.size(0)
    all_scores = []
    for opt_idx in range(5):
        qo_feat = option_feats[:, opt_idx, :]        # (B, 768)
        # ← Simple MLP — no routing!
        fusion_vec = baseline_model(video_feat, qo_feat)  # (B, 512)
        score = scorer(fusion_vec)                         # (B,)
        all_scores.append(score)
    option_scores = torch.stack(all_scores, dim=1)   # (B, 5)
    # ← SIMPLE LOSS: just cross-entropy, no entropy or sparsity terms!
    loss = F.cross_entropy(option_scores, answers)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    preds = option_scores.argmax(dim=1)
    acc = (preds == answers).float().mean().item()
    return loss.item(), acc

In [ ]:
# ── Cell 15: Baseline Validation ───────────────────────────
@torch.no_grad()
def baseline_validate(dataloader, baseline_model, scorer):
    baseline_model.eval()
    scorer.eval()
    total_loss, total_acc, n = 0, 0, 0
    for batch in dataloader:
        video_feat   = batch["video_feat"].to(device)
        option_feats = batch["option_feats"].to(device)
        answers      = batch["answer"].to(device)
        all_scores = []
        for opt_idx in range(5):
            qo_feat = option_feats[:, opt_idx, :]
            fusion_vec = baseline_model(video_feat, qo_feat)
            score = scorer(fusion_vec)
            all_scores.append(score)
        option_scores = torch.stack(all_scores, dim=1)
        loss = F.cross_entropy(option_scores, answers)
        preds = option_scores.argmax(dim=1)
        acc = (preds == answers).float().mean().item()
        total_loss += loss.item()
        total_acc  += acc
        n += 1
    baseline_model.train()
    scorer.train()
    return total_loss / max(n, 1), total_acc / max(n, 1)

In [ ]:
# ── Cell 16: Initialize Baseline Models ────────────────────
baseline_model  = BaselineVQA().to(device)
baseline_scorer = OptionScorer(512).to(device)
print(f"Baseline params:")
print(f"  BaselineVQA  : {sum(p.numel() for p in baseline_model.parameters()):,}")
print(f"  OptionScorer : {sum(p.numel() for p in baseline_scorer.parameters()):,}")
LR = 1e-4
baseline_optimizer = torch.optim.Adam(
    list(baseline_model.parameters()) +
    list(baseline_scorer.parameters()),
    lr=LR, weight_decay=1e-5
)

Baseline params:
  BaselineVQA  : 1,182,208
  OptionScorer : 131,585


In [ ]:
# ── Cell 17: Baseline Training Loop ────────────────────────
EPOCHS    = 30
VAL_EVERY = 5
best_baseline_acc = 0.0
print("=" * 70)
print("  BASELINE Training — Simple Concatenation (No Routing)")
print("=" * 70)
print(f"Epochs: {EPOCHS} | Batch: {BATCH_SIZE} | LR: {LR}")
print("-" * 70)
for epoch in range(EPOCHS):
    baseline_model.train()
    baseline_scorer.train()
    epoch_loss, epoch_acc, n = 0.0, 0.0, 0
    for batch in train_loader:
        loss, acc = baseline_train_step(
            batch, baseline_model, baseline_scorer, baseline_optimizer
        )
        epoch_loss += loss
        epoch_acc  += acc
        n += 1
    avg_loss = epoch_loss / max(n, 1)
    avg_acc  = epoch_acc  / max(n, 1)
    print(f"Epoch {epoch+1:3d}/{EPOCHS} │ Loss: {avg_loss:.4f} │ Acc: {avg_acc:.4f}")
    if (epoch + 1) % VAL_EVERY == 0:
        val_loss, val_acc = baseline_validate(val_loader, baseline_model, baseline_scorer)
        print(f"          ↳ Val Loss: {val_loss:.4f} │ Val Acc: {val_acc:.4f}")
        if val_acc > best_baseline_acc:
            best_baseline_acc = val_acc
            torch.save({
                "baseline": baseline_model.state_dict(),
                "scorer":   baseline_scorer.state_dict(),
                "epoch":    epoch + 1,
                "best_acc": best_baseline_acc,
            }, BASELINE_SAVE)
            print(f"          ↳ ✅ Saved → {BASELINE_SAVE}")
print("-" * 70)
print(f"Baseline best accuracy: {best_baseline_acc:.4f}")

  BASELINE Training — Simple Concatenation (No Routing)
Epochs: 30 | Batch: 4 | LR: 0.0001
----------------------------------------------------------------------
Epoch   1/30 │ Loss: 1.5112 │ Acc: 0.3375
Epoch   2/30 │ Loss: 1.5028 │ Acc: 0.3438
Epoch   3/30 │ Loss: 1.5285 │ Acc: 0.3563
Epoch   4/30 │ Loss: 1.4577 │ Acc: 0.3625
Epoch   5/30 │ Loss: 1.4448 │ Acc: 0.3563
          ↳ Val Loss: 1.6327 │ Val Acc: 0.2000
          ↳ ✅ Saved → /content/drive/MyDrive/dataset/baseline_model_best.pt
Epoch   6/30 │ Loss: 1.4174 │ Acc: 0.3937
Epoch   7/30 │ Loss: 1.3569 │ Acc: 0.4125
Epoch   8/30 │ Loss: 1.3068 │ Acc: 0.4375
Epoch   9/30 │ Loss: 1.2961 │ Acc: 0.4625
Epoch  10/30 │ Loss: 1.3063 │ Acc: 0.4688
          ↳ Val Loss: 1.8775 │ Val Acc: 0.1750
Epoch  11/30 │ Loss: 1.1484 │ Acc: 0.5563
Epoch  12/30 │ Loss: 1.1912 │ Acc: 0.4938
Epoch  13/30 │ Loss: 1.1101 │ Acc: 0.5125
Epoch  14/30 │ Loss: 1.1765 │ Acc: 0.5062
Epoch  15/30 │ Loss: 1.0541 │ Acc: 0.5938
          ↳ Val Loss: 2.2010 │ Val Acc

In [ ]:
# ── Cell 18: FINAL BENCHMARK COMPARISON ────────────────────
# Run this AFTER you have already trained VQAGuider too!
# Replace best_vqaguider_acc with your actual VQAGuider val accuracy!
best_vqaguider_acc = 0.3250   # ← FILL THIS IN from your VQAGuider training output!
print("\n" + "=" * 60)
print("       FINAL BENCHMARK RESULTS")
print("=" * 60)
print(f"  Baseline  (Simple MLP, no routing) : {best_baseline_acc:.4f}  ({best_baseline_acc*100:.1f}%)")
print(f"  VQAGuider (Explicit Neural Routing) : {best_vqaguider_acc:.4f}  ({best_vqaguider_acc*100:.1f}%)")
improvement = (best_vqaguider_acc - best_baseline_acc) * 100
if improvement > 0:
    print(f"\n  ✅ VQAGuider outperforms Baseline by +{improvement:.2f}%!")
    print(f"     This proves Explicit Neural Routing adds real value.")
else:
    print(f"\n  ⚠️  Baseline is currently ahead by {abs(improvement):.2f}%.")
    print(f"     More training epochs or data will close this gap.")
print("=" * 60)


       FINAL BENCHMARK RESULTS
  Baseline  (Simple MLP, no routing) : 0.2500  (25.0%)
  VQAGuider (Explicit Neural Routing) : 0.3250  (32.5%)

  ✅ VQAGuider outperforms Baseline by +7.50%!
     This proves Explicit Neural Routing adds real value.


In [ ]:
# ============================================================
# SIDE-BY-SIDE COMPARISON: Baseline vs VQAGuider
# ============================================================
# Make sure both models are loaded before running this!
# ============================================================
# LOAD SAVED CHECKPOINTS BEFORE COMPARISON
# ============================================================
import torch

# ── Load best VQAGuider (Phase 1) ────────────────────────────
VQAGUIDER_CKPT = "/content/drive/MyDrive/dataset/first/vqa_model_best.pt"
ckpt_vqa = torch.load(VQAGUIDER_CKPT, map_location=device)
vqaguider.load_state_dict(ckpt_vqa["vqaguider"])
scorer.load_state_dict(ckpt_vqa["scorer"])
vqaguider.to(device).eval()
scorer.to(device).eval()
print(f"✅ VQAGuider loaded (epoch {ckpt_vqa.get('epoch','?')}, acc={ckpt_vqa.get('best_acc','?'):.4f})")

# ── Load best Baseline ────────────────────────────────────────
BASELINE_CKPT = "/content/drive/MyDrive/dataset/baseline_model_best.pt"
ckpt_base = torch.load(BASELINE_CKPT, map_location=device)
baseline_model.load_state_dict(ckpt_base["baseline"])
baseline_scorer.load_state_dict(ckpt_base["scorer"])
baseline_model.to(device).eval()
baseline_scorer.to(device).eval()
print(f"✅ Baseline loaded  (epoch {ckpt_base.get('epoch','?')}, acc={ckpt_base.get('best_acc','?'):.4f})")

print("-" * 50)
print("Both models loaded from best saved checkpoints!")
print("Now running comparison on validation set...")
print("-" * 50)

# ── NOW run your comparison code below ────────────────────────


import random

# ── Pick 5 random samples from validation set ───────────────
sample_indices = random.sample(range(len(val_dataset)), min(5, len(val_dataset)))

print("=" * 70)
print("   BASELINE  vs  VQAGuider — Head to Head Comparison")
print("=" * 70)

correct_baseline   = 0
correct_vqaguider  = 0

for i, idx in enumerate(sample_indices):
    video_path, question, options, correct_answer = val_dataset[idx]
    video_id = os.path.splitext(os.path.basename(video_path))[0]

    # ── Get features ──────────────────────────────────────────
    vf = video_features[video_id].unsqueeze(0).to(device)       # (1, 512)
    opt_feats = precomputed_qo[video_id].unsqueeze(0).to(device) # (1, 5, 768)

    # ── Baseline Prediction ───────────────────────────────────
    baseline_model.eval()
    baseline_scorer.eval()
    with torch.no_grad():
        base_scores = []
        for opt_idx in range(5):
            qo = opt_feats[:, opt_idx, :]
            fv = baseline_model(vf, qo)
            base_scores.append(baseline_scorer(fv).item())
    base_pred = int(np.argmax(base_scores))

    # ── VQAGuider Prediction ──────────────────────────────────
    vqaguider.eval()
    scorer.eval()
    with torch.no_grad():
        vqa_scores = []
        task_probs_list = []
        for opt_idx in range(5):
            qo = opt_feats[:, opt_idx, :]
            fv, tp = vqaguider(vf, qo)
            vqa_scores.append(scorer(fv).item())
            task_probs_list.append(tp)
    vqa_pred = int(np.argmax(vqa_scores))
    avg_tp = torch.stack(task_probs_list).mean(dim=0).squeeze().tolist()
    print("RAW task_probs:", task_probs_list)



    # ── Track accuracy ─────────────────────────────────────────
    if base_pred == correct_answer:
        correct_baseline += 1
    if vqa_pred == correct_answer:
        correct_vqaguider += 1

    # ── Print comparison ──────────────────────────────────────
    base_mark = "✅" if base_pred == correct_answer else "❌"
    vqa_mark  = "✅" if vqa_pred  == correct_answer else "❌"

    print(f"\nQ{i+1}: {question}")
    print(f"  Options: {[str(o) for o in options]}")
    print(f"  Correct Answer  : Option {correct_answer} → '{options[correct_answer]}'")
    print(f"  Baseline        : Option {base_pred} → '{options[base_pred]}' {base_mark}")
    print(f"  VQAGuider       : Option {vqa_pred}  → '{options[vqa_pred]}'  {vqa_mark}")
    print(f"  Task Routing    : Action={avg_tp[0]:.2f} | Tracking={avg_tp[1]:.2f} | Scene={avg_tp[2]:.2f}")
    print("-" * 70)

# ── Final Score ───────────────────────────────────────────────
total = len(sample_indices)
print(f"\n  MINI BENCHMARK (on {total} samples)")
print(f"  Baseline  correct: {correct_baseline}/{total} ({correct_baseline/total*100:.0f}%)")
print(f"  VQAGuider correct: {correct_vqaguider}/{total} ({correct_vqaguider/total*100:.0f}%)")
print("=" * 70)


✅ VQAGuider loaded (epoch 15, acc=0.3250)
✅ Baseline loaded  (epoch 25, acc=0.2500)
--------------------------------------------------
Both models loaded from best saved checkpoints!
Now running comparison on validation set...
--------------------------------------------------
   BASELINE  vs  VQAGuider — Head to Head Comparison
RAW task_probs: [tensor([[0.0026, 0.0007, 0.0006]], device='cuda:0'), tensor([[0.0024, 0.0007, 0.0005]], device='cuda:0'), tensor([[0.0030, 0.0009, 0.0007]], device='cuda:0'), tensor([[0.0026, 0.0007, 0.0006]], device='cuda:0'), tensor([[0.0028, 0.0008, 0.0007]], device='cuda:0')]

Q1: what does the cat hold in her mouth
  Options: ['spoon', 'lollipop', 'paper', 'sand', 'watch']
  Correct Answer  : Option 0 → 'spoon'
  Baseline        : Option 0 → 'spoon' ✅
  VQAGuider       : Option 0  → 'spoon'  ✅
  Task Routing    : Action=0.00 | Tracking=0.00 | Scene=0.00
----------------------------------------------------------------------
RAW task_probs: [tensor([[0.0020

In [ ]:
# ── Load Phase 2 projector (generative weights) ───────────────
ckpt_gen = torch.load("/content/drive/MyDrive/dataset/vqa_model_generative.pt", map_location=device)
projector.load_state_dict(ckpt_gen["projector"])
projector.to(device).eval()
print("✅ Phase 2 projector loaded")

def get_vqaguider_generative_answer(video_id, question):
    """Run VQAGuider's full generative pipeline → English answer string."""
    try:
        vf = video_features[video_id].unsqueeze(0).to(device)
        qf = get_question_embedding(question).unsqueeze(0).to(device)

        with torch.no_grad():
            fusion_vec, _ = vqaguider(vf, qf)
            prefix = projector(fusion_vec).to(dtype=torch.float16)  # (1, 10, 2560)

            prompt = (
                f"You are a video understanding AI.\n"
                f"Question: {question}\n"
                f"Detailed Answer:"
            )
            enc = phi2_tokenizer(prompt, return_tensors="pt").to(device)
            token_embeds = phi2.get_input_embeddings()(enc.input_ids).to(dtype=torch.float16)
            inputs_embeds = torch.cat([prefix, token_embeds], dim=1)

            prefix_mask = torch.ones(
                (1, projector.num_tokens),
                dtype=enc.attention_mask.dtype, device=device
            )
            attention_mask = torch.cat([prefix_mask, enc.attention_mask], dim=1)

            output = phi2.generate(
                inputs_embeds=inputs_embeds,
                attention_mask=attention_mask,
                max_new_tokens=30,
                do_sample=False,
                pad_token_id=phi2_tokenizer.eos_token_id,
            )

        answer = phi2_tokenizer.decode(output[0], skip_special_tokens=True)
        answer = answer.split("Detailed Answer:")[-1].strip()
        answer = answer.split("\n")[0].strip()
        return answer

    except Exception as e:
        return f"Error: {str(e)[:50]}"


In [ ]:
# ============================================================
# AUTOMATED BENCHMARK EVALUATOR → Excel Report
# ============================================================
import pandas as pd
from tqdm import tqdm

# ── Configuration ─────────────────────────────────────────────
EXCEL_SAVE_PATH = "/content/drive/MyDrive/dataset/benchmark_results.xlsx"
NUM_SAMPLES = len(val_dataset)   # ← Evaluate on ALL validation samples

# ── Load best checkpoints ─────────────────────────────────────
ckpt_vqa  = torch.load("/content/drive/MyDrive/dataset/first/vqa_model_best.pt",      map_location=device)
ckpt_base = torch.load("/content/drive/MyDrive/dataset/baseline_model_best.pt", map_location=device)

vqaguider.load_state_dict(ckpt_vqa["vqaguider"])
scorer.load_state_dict(ckpt_vqa["scorer"])
vqaguider.to(device).eval()
scorer.to(device).eval()

baseline_model.load_state_dict(ckpt_base["baseline"])
baseline_scorer.load_state_dict(ckpt_base["scorer"])
baseline_model.to(device).eval()
baseline_scorer.to(device).eval()

print(f"✅ VQAGuider loaded  (best acc: {ckpt_vqa.get('best_acc', '?')})")
print(f"✅ Baseline loaded   (best acc: {ckpt_base.get('best_acc', '?')})")
print(f"📊 Evaluating {NUM_SAMPLES} samples from validation set...")

# ── Run Evaluation ────────────────────────────────────────────
rows = []

with torch.no_grad():
    for idx in tqdm(range(NUM_SAMPLES), desc="Evaluating"):
        video_path, question, options, correct_answer = val_dataset[idx]
        video_id = os.path.splitext(os.path.basename(video_path))[0]

        # Get pre-computed features
        vf       = video_features[video_id].unsqueeze(0).to(device)        # (1, 512)
        opt_feats = precomputed_qo[video_id].unsqueeze(0).to(device)       # (1, 5, 768)

        # ── Baseline Prediction ───────────────────────────────
        base_scores = []
        for opt_idx in range(5):
            qo = opt_feats[:, opt_idx, :]
            fv = baseline_model(vf, qo)
            base_scores.append(baseline_scorer(fv).item())
        base_pred = int(np.argmax(base_scores))

        # ── VQAGuider Prediction ──────────────────────────────
        vqa_scores = []
        task_probs_all = []
        for opt_idx in range(5):
            qo = opt_feats[:, opt_idx, :]
            fv, tp = vqaguider(vf, qo)
            vqa_scores.append(scorer(fv).item())
            task_probs_all.append(tp.squeeze().tolist())
        vqa_pred = int(np.argmax(vqa_scores))

        # Average task probs across all 5 options
        avg_action   = np.mean([t[0] for t in task_probs_all])
        avg_tracking = np.mean([t[1] for t in task_probs_all])
        avg_scene    = np.mean([t[2] for t in task_probs_all])

        # ── Build Row ─────────────────────────────────────────
        rows.append({
            "Video ID"              : video_id,
            "Question"              : question,
            "Option A"              : str(options[0]),
            "Option B"              : str(options[1]),
            "Option C"              : str(options[2]),
            "Option D"              : str(options[3]),
            "Option E"              : str(options[4]),
            "Correct Answer (idx)"  : correct_answer,
            "Correct Answer (text)" : str(options[correct_answer]),

            # Baseline results
            "Baseline Predicted (idx)"  : base_pred,
            "Baseline Predicted (text)" : str(options[base_pred]),
            "Baseline Correct?"         : "✅ YES" if base_pred == correct_answer else "❌ NO",

            # VQAGuider results
            "VQAGuider Predicted (idx)"  : vqa_pred,
            "VQAGuider Predicted (text)" : str(options[vqa_pred]),
            "VQAGuider Correct?"         : "✅ YES" if vqa_pred == correct_answer else "❌ NO",

            # Task routing
            "Task: Action"   : round(avg_action,   4),
            "Task: Tracking" : round(avg_tracking, 4),
            "Task: Scene"    : round(avg_scene,    4),
        })

# ── Build DataFrame ───────────────────────────────────────────
df = pd.DataFrame(rows)

# ── Summary Statistics Row ────────────────────────────────────
base_acc  = (df["Baseline Correct?"] == "✅ YES").sum() / len(df) * 100
vqa_acc   = (df["VQAGuider Correct?"] == "✅ YES").sum() / len(df) * 100

print("\n" + "=" * 60)
print("       BENCHMARK SUMMARY")
print("=" * 60)
print(f"  Total samples evaluated : {len(df)}")
print(f"  Baseline  accuracy      : {base_acc:.2f}%")
print(f"  VQAGuider accuracy      : {vqa_acc:.2f}%")
print(f"  VQAGuider improvement   : +{vqa_acc - base_acc:.2f}%")
print("=" * 60)

# ── Save to Excel ─────────────────────────────────────────────
with pd.ExcelWriter(EXCEL_SAVE_PATH, engine="openpyxl") as writer:

    # Sheet 1: Full results
    df.to_excel(writer, sheet_name="Full Results", index=False)

    # Sheet 2: Only where models disagree (most interesting!)
    df_disagree = df[
        df["Baseline Predicted (idx)"] != df["VQAGuider Predicted (idx)"]
    ]
    df_disagree.to_excel(writer, sheet_name="Model Disagreements", index=False)

    # Sheet 3: Summary stats
    summary = pd.DataFrame([{
        "Model"          : "Baseline (Simple MLP)",
        "Correct"        : (df["Baseline Correct?"] == "✅ YES").sum(),
        "Wrong"          : (df["Baseline Correct?"] == "❌ NO").sum(),
        "Accuracy (%)"   : round(base_acc, 2),
    }, {
        "Model"          : "VQAGuider (Expert Routing)",
        "Correct"        : (df["VQAGuider Correct?"] == "✅ YES").sum(),
        "Wrong"          : (df["VQAGuider Correct?"] == "❌ NO").sum(),
        "Accuracy (%)"   : round(vqa_acc, 2),
    }])
    summary.to_excel(writer, sheet_name="Summary", index=False)

print(f"\n✅ Excel report saved → {EXCEL_SAVE_PATH}")
print(f"   Sheet 1: Full Results      ({len(df)} rows)")
print(f"   Sheet 2: Model Disagreements ({len(df_disagree)} rows)")
print(f"   Sheet 3: Summary Statistics")
